# Week 1: Video / Transcript → Text → Summarization

This notebook shows **video-to-text** (via transcript) then **summarization** with an LLM. You can use a YouTube transcript or a local transcript file.

**Concepts:** Get transcript (video-to-text), load prompt template, call API, display summary.

**Prerequisites:** [Week 1 guide](docs/weeks/Week1.md), `.env` with `OPENAI_API_KEY`, `uv sync` from repo root.

**Run order:** Execute cells from top to bottom.

## 1. Setup and imports

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dotenv import load_dotenv
load_dotenv(repo_root / ".env")

from scripts.api_client import get_openai_client

print("Repo root:", repo_root)
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

## 2. Get transcript (video-to-text)

Option A: Fetch transcript from a public YouTube video (requires `youtube-transcript-api`).  
Option B: Use a short sample transcript below if you don't want to call YouTube.

In [ ]:
def get_transcript_from_youtube(video_id: str, max_chars: int = 4000) -> str:
    """Fetch captions for a public YouTube video and return as one string."""
    try:
        from youtube_transcript_api import YouTubeTranscriptApi
    except ImportError:
        return ""
    try:
        items = YouTubeTranscriptApi.get_transcript(video_id)
        text = " ".join(item["text"] for item in items)
        return text[:max_chars] if len(text) > max_chars else text
    except Exception:
        return ""

# Example: use a short sample transcript if you skip YouTube
SAMPLE_TRANSCRIPT = """
Today we're talking about large language models. First, what is an LLM? It's a model trained on huge amounts of text. You give it a prompt and it generates text. Engineers use APIs to call these models from code. Common use cases include summarization, question answering, and chatbots. Always keep your API keys in environment variables, never in code.
"""

# Option A: YouTube (replace with a real public video ID, or leave empty to use sample)
VIDEO_ID = ""  # e.g. "dQw4w9WgXcQ"
if VIDEO_ID:
    transcript = get_transcript_from_youtube(VIDEO_ID)
    if not transcript:
        transcript = SAMPLE_TRANSCRIPT
        print("Using sample transcript (YouTube failed or no captions).")
    else:
        print(f"Fetched transcript: {len(transcript)} chars")
else:
    transcript = SAMPLE_TRANSCRIPT.strip()
    print("Using sample transcript. Set VIDEO_ID to a YouTube video ID to use real captions.")

print("First 200 chars:", transcript[:200], "..." if len(transcript) > 200 else "")

## 3. Load prompt and call LLM

In [ ]:
prompt_path = repo_root / "prompts" / "video-transcript-summarize.prompt.txt"
template = prompt_path.read_text(encoding="utf-8")
user_content = template.replace("{{content}}", transcript)

client = get_openai_client()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": user_content}],
    temperature=0.2,
    max_tokens=400,
)
summary = response.choices[0].message.content
print("--- Summary ---")
print(summary)